# Quran Word Frequencies — Results & Exploration

**Source**: Quranic Arabic Corpus morphology v0.4 (77,915 STEM entries)  
**Method**: Buckwalter lemma matching against corpus LEM field

> **Audited 2026-06-11** — see `03_audit_and_full_recount.ipynb` for the full audit and method
> grid. Corrections applied here: the embryology Izam count now includes the plural عظام tagged
> under `LEM:EaZiym` (2 → 15), and the pair notes carry corrected Barr (12 land / 10 righteous)
> and symmetric Hayat/Mawt variant numbers (79 / 56).

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
from parser import load_morphology, load_prefixes
from buckwalter import bw_to_arabic

df = load_morphology()
lem_counts = df["LEM"].value_counts()

def lc(lem, gender=None):
    """Count a lemma, with optional gender filter on PGN."""
    mask = df["LEM"] == lem
    if gender:
        mask = mask & df["PGN"].str.contains(gender, na=False)
    return int(mask.sum())

print(f"Loaded {len(df):,} stems, {df['LEM'].nunique():,} lemmas, {df['ROOT'].nunique():,} roots")

Loaded 77,915 stems, 4,832 lemmas, 1,642 roots


## 1. The 15 original pairs — validated counts

In [2]:
# All 15 pairs from the original screenshots
# Format: (pair_name, lemma_a, gender_filter_a, lemma_b, gender_filter_b, notes)
PAIRS = [
    ("Dunya / Akhira",   "d~unoyaA", None, "A^xir",      "F",   "Akhira fem only = 'hereafter'. Masc (40x) = 'last' (different word)."),
    ("Malak / Shaytan",  "malak",    None, "$ayoTa`n",   None,  "Both lemmas include their plurals (73 vs 18)."),
    ("Adam / Isa",       "A^dam",    None, "EiysaY",     None,  "Proper nouns, no root."),
    ("Hayat / Mawt",     "Hayaw`p",  None, "mawot",      None,  "With variants (symmetric): Hayat +maHyaA,HayawaAn = 79; Mawt +mawotat,mamaAt = 56."),
    ("Rajul / Imra'a",   "rajul",    None, "{mora>at",   None,  "With plurals: 57 vs 85."),
    ("Jannah / Jahannam","jan~ap",   None, "jahan~am",   None,  ""),
    ("Hasana / Sayyi'a", "Hasanap",  None, "say~i}ap",   None,  "With plurals: 31 vs 58."),
    ("Iman / Kufr",      "<iyma`n",  None, "kufor",      None,  "Kufr +variants = 41."),
    ("Rahma / Adhab",    "raHomap",  None, "Ea*aAb",     None,  ""),
    ("Ghani / Faqir",    "ganiY~",   None, "faqiyr",     None,  "Exact 2:1 ratio."),
    ("Bahr / Barr",      "baHor",    None, "bar~",       None,  "Barr conflates land(12) + righteous/dutiful(10) — form-level split in notebook 03."),
    ("Harr / Bard",      "Har~",     None, "barod",      None,  "Small counts. Harr + Haruwr = 4; Bard + adj baArid = 4."),
    ("Zakat / Baraka",   "zakaw`p",  None, "baraka`t",   None,  "Baraka + blessed adj (mubaArak, mubarakap) = 15."),
    ("Insan / Iblis",    "<insa`n",  None, "<iboliys",   None,  "Insan + ins (collective noun) = 89."),
    ("Shahr / Yawm",     "$ahor",    None, "yawom",      None,  "Singular-only: Shahr 12, Yawm 375."),
]

rows = []
for name, la, ga, lb, gb, notes in PAIRS:
    ca, cb = lc(la, ga), lc(lb, gb)
    rows.append({
        "Pair": name,
        "Arabic A": bw_to_arabic(la),
        "Count A": ca,
        "": "=" if ca == cb else "",
        "Count B": cb,
        "Arabic B": bw_to_arabic(lb),
        "Notes": notes,
    })

pairs_df = pd.DataFrame(rows)
display(pairs_df[["Pair", "Arabic A", "Count A", "", "Count B", "Arabic B"]])

print("\nExact matches:")
for _, r in pairs_df[pairs_df[""] == "="].iterrows():
    print(f"  {r['Pair']} = {r['Count A']}")

print("\nNotes:")
for _, r in pairs_df[pairs_df["Notes"] != ""].iterrows():
    print(f"  {r['Pair']}: {r['Notes']}")

,Pair,Arabic A,Count A,,Count B,Arabic B
0,Dunya / Akhira,دُّنْيَا,115,=,115,ا^خِر
1,Malak / Shaytan,مَلَك,88,=,88,شَيْطَٰن
2,Adam / Isa,ا^دَم,25,=,25,عِيسَى
3,Hayat / Mawt,حَيَوٰة,76,,50,مَوْت
4,Rajul / Imra'a,رَجُل,29,,26,ٱمْرَأَت
5,Jannah / Jahannam,جَنَّة,147,,77,جَهَنَّم
6,Hasana / Sayyi'a,حَسَنَة,28,,22,سَيِّئَة
7,Iman / Kufr,إِيمَٰن,45,,37,كُفْر
8,Rahma / Adhab,رَحْمَة,114,,322,عَذَاب
9,Ghani / Faqir,غَنِىّ,24,,12,فَقِير



Exact matches:
  Dunya / Akhira = 115
  Malak / Shaytan = 88
  Adam / Isa = 25

Notes:
  Dunya / Akhira: Akhira fem only = 'hereafter'. Masc (40x) = 'last' (different word).
  Malak / Shaytan: Both lemmas include their plurals (73 vs 18).
  Adam / Isa: Proper nouns, no root.
  Hayat / Mawt: With variants (symmetric): Hayat +maHyaA,HayawaAn = 79; Mawt +mawotat,mamaAt = 56.
  Rajul / Imra'a: With plurals: 57 vs 85.
  Hasana / Sayyi'a: With plurals: 31 vs 58.
  Iman / Kufr: Kufr +variants = 41.
  Ghani / Faqir: Exact 2:1 ratio.
  Bahr / Barr: Barr conflates land(12) + righteous/dutiful(10) — form-level split in notebook 03.
  Harr / Bard: Small counts. Harr + Haruwr = 4; Bard + adj baArid = 4.
  Zakat / Baraka: Baraka + blessed adj (mubaArak, mubarakap) = 15.
  Insan / Iblis: Insan + ins (collective noun) = 89.
  Shahr / Yawm: Singular-only: Shahr 12, Yawm 375.


## 2. Standalone words + Embryology sequence

In [3]:
# Standalone words
print("Standalone words:\n")
print(f"  Qaala (قال, said):        {lc('qaAla'):>5}")
print(f"  Maghfira (مغفرة, forgiveness): {lc('m~agofirap'):>5}  — paired with what in screenshots?")

# Embryology sequence
print("\n\nEmbryology sequence:\n")
# AUDIT CORRECTION: the plural عظام (bones) is tagged LEM:EaZiym + POS:N + MP in the corpus
# (not under EaZom), so Izam needs an extra selector — see notebook 03, section 2.1.
izam_pl = int(((df["LEM"] == "EaZiym") & (df["POS"] == "N") & (df["NUMBER"] == "P")).sum())
embryo = [
    ("Turab (تراب, dust)",      lc("turaAb")),
    ("Nutfa (نطفة, drop)",      lc("n~uTofap")),
    ("Alaqa (علقة, clot)",      lc("Ealaqap")),
    ("Alaqa variant (علق)",     lc("Ealaq")),
    ("Mudgha (مضغة, lump)",     lc("muDogap")),
    ("Izam (عظم, bone sg.)",    lc("EaZom")),
    ("Izam (عظام, bones pl.)",  izam_pl),
    ("Lahm (لحم, flesh)",       lc("laHom")),
]
total = 0
for name, c in embryo:
    total += c
    print(f"  {name:<30} = {c:>3}")
print(f"  {'TOTAL':<30} = {total:>3}")

Standalone words:

  Qaala (قال, said):         1618
  Maghfira (مغفرة, forgiveness):    28  — paired with what in screenshots?


Embryology sequence:

  Turab (تراب, dust)             =  17
  Nutfa (نطفة, drop)             =  12
  Alaqa (علقة, clot)             =   5
  Alaqa variant (علق)            =   1
  Mudgha (مضغة, lump)            =   3
  Izam (عظم, bone sg.)           =   2
  Izam (عظام, bones pl.)         =  13
  Lahm (لحم, flesh)              =  12
  TOTAL                          =  65


---
## 3. Open-ended exploration

Everything below is exploring the raw data for patterns we haven't specifically looked for.

### 3.1 — Top 30 most frequent lemmas

In [4]:
top30 = lem_counts.head(30).reset_index()
top30.columns = ["BW_Lemma", "Count"]
top30["Arabic"] = top30["BW_Lemma"].apply(bw_to_arabic)
top30["POS"] = top30["BW_Lemma"].apply(lambda l: df[df["LEM"] == l]["POS"].mode().iloc[0])
top30.index = range(1, 31)
display(top30[["Arabic", "BW_Lemma", "POS", "Count"]])

,Arabic,BW_Lemma,POS,Count
1,مِن,min,P,3226
2,ٱللَّه,{ll~ah,PN,2699
3,مَا,maA,REL,2565
4,لَا,laA,NEG,1738
5,فِى,fiY,P,1701
6,إِنّ,<in~,ACC,1682
7,قَالَ,qaAla,V,1618
8,ٱلَّذِى,{l~a*iY,REL,1464
9,عَلَىٰ,EalaY`,P,1445
10,كَانَ,kaAna,V,1358


### 3.2 — All proper nouns + coincidences

In [5]:
pn = df[df["POS"] == "PN"]["LEM"].value_counts().reset_index()
pn.columns = ["BW_Lemma", "Count"]
pn["Arabic"] = pn["BW_Lemma"].apply(bw_to_arabic)
print(f"All {len(pn)} proper nouns in the Quran:\n")
display(pn[["Arabic", "BW_Lemma", "Count"]])

print("\n\nProper nouns with identical counts:\n")
pn_counts = df[df["POS"] == "PN"]["LEM"].value_counts()
pn_by_count = pn_counts.groupby(pn_counts.values).apply(lambda g: list(g.index))
for count_val in sorted(pn_by_count.index, reverse=True):
    names = pn_by_count[count_val]
    if len(names) >= 2:
        arabic_list = [f"{bw_to_arabic(n)}" for n in names]
        print(f"  Count = {count_val:>3}: {', '.join(arabic_list)}")

All 107 proper nouns in the Quran:



,Arabic,BW_Lemma,Count
0,ٱللَّه,{ll~ah,2699
1,مُوسَىٰ,muwsaY`,136
2,شَيْطَٰن,$ayoTa`n,80
3,جَهَنَّم,jahan~am,77
4,فِرْعَوْن,firoEawon,74
...,...,...,...
102,نَسْر,nasor,1
103,سَلْسَبِيل,salosabiyl,1
104,إِرَم,<iram,1
105,سِينِين,siyniyn,1




Proper nouns with identical counts:

  Count =  43: إِسْرَائِيل, نُوح
  Count =  27: يُوسُف, لُوط
  Count =  25: ا^دَم, عِيسَى
  Count =  17: سُلَيْمَٰن, إِسْحَاق
  Count =  16: يَعْقُوب, دَاوُد
  Count =  12: إِسْمَاعِيل, إِنجِيل
  Count =  11: إِبْلِيس, مَسِيح, شُعَيْب, عَدْن
  Count =   9: يَهُودِيّ, صَٰلِح2
  Count =   7: زَكَرِيَّا, هُود
  Count =   6: إِسْلَٰم, هَٰمَٰن
  Count =   5: ٱللَّهُمَّ, يَحْيَىٰ
  Count =   4: مُحَمَّد, أَيُّوب, يُونُس, مَدِينَة, مِصْر, قَٰرُون, سَقَر
  Count =   3: صَّٰبِـ#ِين, جِبْرِيل, هُود2, جَالُوت, عِمْرَٰن, زَبُور, جَٰهِلِيَّة2, إِلْيَاس, سَّامِرِىّ, زَقُّوم
  Count =   2: طَالُوت, كَعْبَة, ٱلْيَسَعَ, يَأْجُوج, مَأْجُوج, فِرْدَوْس, إِدْرِيس, سَبَإ, لُقْمَٰن, تُبَّع
  Count =   1: مِيكَىٰل, بَابِل, هَٰرُوت, مَٰرُوت, صَّفَا, مَرْوَة, رَمَضَان, عَرَفَٰت, سِّلْم, بَكَّة, بَدْر, ا^زَر, حُنَيْن, عُزَيْر, جُودِىّ, مَجُوس, سَيْنَا^ء, رُّوم, يَثْرِب, زَيْد, بَعْل2, مَٰلِك2, أَحْقَاف, مَكَّة, ٱللَّٰت, ٱلْعُزَّىٰ, مَنَوٰة, شِّعْرَىٰ, أَحْمَد, جُمُعَة, لَظَ

### 3.3 — Broader semantic pairs (beyond the original 15)

In [6]:
# Check a broader set of antonym/related pairs
MORE_PAIRS = [
    ("layl (night)",     "layol",    None,  "nahar (daytime)",     "nahaAr",   None),
    ("samaa (sky)",      "samaA^'",  None,  "ard (earth)",         ">aroD",    None),
    ("nuur (light)",     "nuwr",     None,  "zulumat (darkness)",  "Zuluma`t", None),
    ("haqq (truth)",     "Haq~",     None,  "batil (falsehood)",   "ba`Til",   None),
    ("khayr (good)",     "xayor",    None,  "sharr (evil)",        "$ar~",     None),
    ("shams (sun)",      "$amos",    None,  "qamar (moon)",        "qamar",    None),
    ("maa (water)",      "maA^'",    None,  "naar (fire)",         "naAr",     None),
    ("mashriq (east)",   "ma$oriq",  None,  "maghrib (west)",      "magorib",  None),
    ("salaam (peace)",   "sala`m",   None,  "harb (war)",          "Harob",    None),
    ("qariib (near)",    "qariyb",   None,  "ba'iid (far)",        "baEiyd",   None),
    ("ab (father)",      ">abN",     None,  "umm (mother)",        ">um~",     None),
    ("kafiroon (disb.)", "ka`firuwn",None,  "zalim (wrongdoer)",   "ZaAlim",   None),
    ("walad (child)",    "walad",    None,  "mawt (death)",        "mawot",    None),
    ("samee' (hearing)", "samiyE",   None,  "baseer (seeing)",     "baSiyr",   None),
]

rows = []
for na, la, ga, nb, lb, gb in MORE_PAIRS:
    ca, cb = lc(la, ga), lc(lb, gb)
    eq = "=" if ca == cb else ("~" if abs(ca-cb) <= 2 and min(ca,cb) > 0 else "")
    rows.append({"Word A": na, "Arabic A": bw_to_arabic(la), "#A": ca,
                 "": eq, "#B": cb, "Arabic B": bw_to_arabic(lb), "Word B": nb})

more_df = pd.DataFrame(rows)
display(more_df)

exact = more_df[more_df[""] == "="]
near = more_df[more_df[""] == "~"]
if len(exact):
    print(f"\nExact matches: {len(exact)}")
    for _, r in exact.iterrows():
        print(f"  {r['Word A']} = {r['Word B']} = {r['#A']}")
if len(near):
    print(f"\nNear matches (within 2):")
    for _, r in near.iterrows():
        print(f"  {r['Word A']} ({r['#A']}) ~ {r['Word B']} ({r['#B']})")

,Word A,Arabic A,#A,,#B,Arabic B,Word B
0,layl (night),لَيْل,84,,57,نَهَار,nahar (daytime)
1,samaa (sky),سَمَا^ء,310,,461,أَرْض,ard (earth)
2,nuur (light),نُور,43,,23,ظُلُمَٰت,zulumat (darkness)
3,haqq (truth),حَقّ,247,,26,بَٰطِل,batil (falsehood)
4,khayr (good),خَيْر,178,,30,شَرّ,sharr (evil)
5,shams (sun),شَمْس,33,,27,قَمَر,qamar (moon)
6,maa (water),مَا^ء,63,,145,نَار,naar (fire)
7,mashriq (east),مَشْرِق,11,~,10,مَغْرِب,maghrib (west)
8,salaam (peace),سَلَٰم,42,,4,حَرْب,harb (war)
9,qariib (near),قَرِيب,26,~,25,بَعِيد,ba'iid (far)



Exact matches: 1
  kafiroon (disb.) = zalim (wrongdoer) = 129

Near matches (within 2):
  mashriq (east) (11) ~ maghrib (west) (10)
  qariib (near) (26) ~ ba'iid (far) (25)


### 3.4 — Notable numbers: do any lemma counts match structurally significant numbers?

In [7]:
notable = {
    7: "days of week / heavens",
    12: "months",
    19: "Quran numerology (74:30)",
    30: "days in month",
    99: "names of Allah",
    114: "surahs in Quran",
    313: "warriors at Badr (tradition)",
    354: "days in lunar year",
    365: "days in solar year",
}

print("Lemmas whose count matches a notable number:\n")
for num, meaning in sorted(notable.items()):
    matching = lem_counts[lem_counts == num]
    if len(matching) > 0:
        words = [f"{bw_to_arabic(l)} ({l})" for l in matching.index[:5]]
        extra = f" +{len(matching)-5} more" if len(matching) > 5 else ""
        print(f"  {num:>3} ({meaning}): {', '.join(words)}{extra}")
    else:
        print(f"  {num:>3} ({meaning}): —")

print(f"\nNotable: Rahma (رحمة, mercy) appears exactly 114 times = number of surahs")

Lemmas whose count matches a notable number:

    7 (days of week / heavens): سَفِيه (safiyh), يَعْمَهُ (yaEomahu), كَبِيرَة (kabiyrap), أَرْبَع (>arobaE), فُرْقَان (furoqaAn) +97 more
   12 (months): يُوقِنُ (yuwqinu), أَصَمّ (>aSam~), قُطِعَ (quTiEa), تَوَّاب (taw~aAb), أَدْنَىٰ (>adonaY`) +45 more
   19 (Quran numerology (74:30)): شَجَرَة ($ajarap), مُصَدِّق (muSad~iq), مُّعْرِضُون (m~uEoriDuwn), يَضُرَّ (yaDur~a), أَيْن (>ayon) +10 more
   30 (days in month): وَٰحِد (wa`Hid), قُوَّة (quw~ap), وَلَّىٰ (wal~aY`), عَسَى (EasaY), شَرّ ($ar~) +5 more
   99 (names of Allah): أَخْرَجَ (>axoraja)
  114 (surahs in Quran): رَحْمَة (raHomap)
  313 (warriors at Badr (tradition)): —
  354 (days in lunar year): —
  365 (days in solar year): —

Notable: Rahma (رحمة, mercy) appears exactly 114 times = number of surahs


### 3.5 — All lemma count coincidences (count >= 10, 2+ lemmas sharing same count)

This is the raw data — every group of lemmas that share the same frequency.

In [8]:
freq_above_10 = lem_counts[lem_counts >= 10]
count_groups = freq_above_10.groupby(freq_above_10.values).apply(lambda g: list(g.index))

print("Lemma count coincidences (count >= 10, 2+ lemmas):\n")
for count_val in sorted(count_groups.index, reverse=True):
    lemmas = count_groups[count_val]
    if len(lemmas) >= 2:
        arabic_list = [f"{bw_to_arabic(l)}" for l in lemmas]
        print(f"  {count_val:>4}: {', '.join(arabic_list)}")

Lemma count coincidences (count >= 10, 2+ lemmas):

   382: عَلِمَ, ءَايَة
   271: ا^تَى, رَءَا
   176: كَذَّبَ, سَبِيل
   166: ٱتَّقَىٰ, أَمْر
   147: غَيْر, جَنَّة, إِلَٰه
   144: هَدَى, دُون
   136: مُوسَىٰ, ٱتَّبَعَ
   129: كَٰفِرُون, ظَالِم
   127: أَخَذَ, بَل, أَهْل
   120: عَظِيم, يَد
   106: لَن, سَأَلَ, وَجَدَ
   105: عِلْم, أَجْر
    93: أَكَلَ, ذُو, هَل
    92: دِين, قَوْل
    88: شَيْطَٰن, مَثَل, فَعَلَ, مَلَك
    86: وَلِىّ, مَال
    84: ذَكَرَ, فَضْل, لَيْل
    83: صَلَوٰة, كَيْف, قَتَلَ, خَافَ
    80: بُنَىّ, أَكْثَر
    78: أَصْحَٰب, تَوَلَّىٰ, سَمِعَ
    77: أَمَرَ, جَهَنَّم
    76: زَوْج, دَخَلَ, حَيَوٰة, ذِكْر
    75: مِثْل, نَّبِىّ, لَوْلَا^, أَخ
    74: خَٰلِد, فِرْعَوْن, أَحَد
    73: عَٰلَمِين, لَٰكِن, جَزَىٰ
    72: أَلِيم, وَجْه, أَطَاعَ, أَوْحَىٰ^
    71: إِنسَٰن, بَيِّنَة, أَشْرَكَ, عَمَل, أَلْقَىٰ^
    70: قَلِيل, قِيَٰمَة, ا^خَر, قُرْءَان, وَعَدَ, يَوْمَئِذ
    65: غَفَرَ, صَٰلِح, بَيْت, يَمِين
    64: أَضَلَّ, أُمَّة, ا^بَاء, أَصَابَ, أَحْبَبْ
    63: مَا^

### 3.6 — Verse co-occurrence: which content words appear together most?

In [9]:
# For top content words, which pairs co-occur in the same verse most?
content_lems = df[df["POS"].isin(["N", "V", "ADJ"])]["LEM"].value_counts().head(30).index.tolist()

verse_sets = {}
for lem in content_lems:
    verse_sets[lem] = set(zip(df[df["LEM"] == lem]["chapter"], df[df["LEM"] == lem]["verse"]))

cooc = []
for i, la in enumerate(content_lems):
    for lb in content_lems[i+1:]:
        shared = len(verse_sets[la] & verse_sets[lb])
        smaller = min(len(verse_sets[la]), len(verse_sets[lb]))
        if smaller > 0 and shared > 5:
            cooc.append((la, lb, shared, shared/smaller, smaller))

cooc.sort(key=lambda x: -x[3])

print("Top 15 content word pairs by co-occurrence rate:\n")
print(f"  {'Word A':<12} {'Word B':<12} {'Shared':>6} {'Rate':>6} {'Min':>5}")
print(f"  {'-'*50}")
for la, lb, shared, rate, minc in cooc[:15]:
    print(f"  {bw_to_arabic(la):<12} {bw_to_arabic(lb):<12} {shared:>6} {rate:>5.0%} {minc:>5}")

Top 15 content word pairs by co-occurrence rate:

  Word A       Word B       Shared   Rate   Min
  --------------------------------------------------
  أَرْض        سَمَا^ء         222   75%   297
  كُلّ         شَىْء           120   45%   266
  قَالَ        جَا^ءَ          116   44%   262
  كَانَ        قَبْل            95   40%   235
  كَانَ        مُؤْمِن          77   40%   193
  قَالَ        قَوْم           137   39%   347
  قَالَ        رَءَا            98   39%   253
  قَالَ        أَتَى            92   38%   240
  كَانَ        عَمِلَ           96   38%   255
  قَالَ        رَبّ            320   37%   866
  كَانَ        نَفْس            86   32%   268
  قَالَ        قَبْل            74   31%   235
  قَالَ        كَفَرَ           84   31%   269
  أَرْض        خَلَقَ           51   30%   168
  قَالَ        كَانَ           348   30%  1149
